In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from index.inverted_index import Index
from index.documents import load_dataset, tokenizer

msmarco = load_dataset("msmarco-passage/dev/judged")

In [3]:
import locale, sys
print(sys.flags.utf8_mode)                 # should be 1
print(locale.getpreferredencoding(False))  # should be UTF-8

1
utf-8


In [4]:
NUM_DOCS = -1
K = 10

## Shuffle the documents!

In [ ]:
from time import perf_counter

msmarco_index = Index()
msmarco_index.clear_index()

load_time = perf_counter()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=NUM_DOCS)
load_time_end = perf_counter()

print(f"Indexing time: {load_time_end - load_time:.2f} seconds")

compression_start = perf_counter()
msmarco_index.compress_index(method="gamma")
compression_end = perf_counter()

print(f"Compression time: {compression_end - compression_start:.2f} seconds")

save_start = perf_counter()
msmarco_index.save_index("msmarco_index.bin")
save_end = perf_counter()

print(f"Save time: {save_end - save_start:.2f} seconds")

decompression_start = perf_counter()
msmarco_index.decompress_index()
decompression_end = perf_counter()

print(f"Decompression time: {decompression_end - decompression_start:.2f} seconds")

loaded_index = Index()
loaded_index.clear_index()

load_start = perf_counter()
loaded_index.load_index("msmarco_index.bin")
load_end = perf_counter()

print(f"Load time with gamma decompression: {load_end - load_start:.2f} seconds")

del loaded_index
del msmarco_index

Processing:   23.1% [2039000/8841823] | Current docid: 2039000             

In [ ]:
def check_correctness(original_index, loaded_index):
    for term in original_index.index.keys():
        original_postings = original_index.index[term]
        loaded_postings = loaded_index.index.get(term, [])
        
        if original_postings != loaded_postings:
            print(f"Mismatch for term '{term}':")
            print(f"Original: {original_postings}")
            print(f"Loaded: {loaded_postings}")
            return False
    return True

In [ ]:
msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=NUM_DOCS)

## Sparse tf_idf Matrix

In [ ]:
from scipy.sparse import csr_matrix
import numpy as np
import pandas as pd

def create_sparse_tf_idf_matrix(index, dtype=np.float32):
    terms = list(index.get_terms())
    term_to_col = {t:i for i,t in enumerate(terms)}
    doc_ids = index.get_doc_ids()
    doc_to_row = {did:i for i,did in enumerate(doc_ids)}

    rows, cols, data = [], [], []
    for term, posting in index.index.items():
        idf = index.get_inverse_document_frequency(term)
        col = term_to_col[term]
        for doc_id, tf in posting:
            rows.append(doc_to_row[doc_id])
            cols.append(col)
            data.append(np.log10(1+tf) * idf)

    data = np.array(data, dtype=dtype)
    mat = csr_matrix((data, (rows, cols)), shape=(len(doc_ids), len(terms)), dtype=dtype)
    return mat, doc_ids, terms

In [ ]:
mat, doc_ids, terms = create_sparse_tf_idf_matrix(unordered_index)
df = pd.DataFrame.sparse.from_spmatrix(mat, index=doc_ids, columns=terms)

In [ ]:
# term_index = {term: idx for idx, term in enumerate(terms)}
# # Transform each document into a set of termIDs

# doc_set = {doc_id: set(term_index[term] for term in msmarco.docs_iter()[1] f ) for doc_id in doc_ids}

## document-term mapping

In [32]:

doc_id_term_id = unordered_index.compute_docId_termId(msmarco, tokenizer, max_num_docs=NUM_DOCS)

In [33]:
# Reorder the documents according to the size of document set (reverse order)

# doc_ids = sorted(doc_id_term_id, key=lambda x: len(doc_id_term_id.get(x)), reverse=True)


doc_id_term_id = {
    doc_id: doc_id_term_id[doc_id]  # take the document
    for doc_id in sorted(doc_id_term_id, key=lambda x: len(doc_id_term_id.get(x)), reverse=True) # 
}

"""
why we are sorting?
Nel tuo algoritmo di clustering, il primo documento che incontri diventa un medoide.
Se processi prima i documenti più "ricchi" (quelli con più termini), hai più probabilità che diventino dei buoni rappresentanti dei cluster.
"""



'\nwhy we are sorting?\nNel tuo algoritmo di clustering, il primo documento che incontri diventa un medoide.\nSe processi prima i documenti più "ricchi" (quelli con più termini), hai più probabilità che diventino dei buoni rappresentanti dei cluster.\n'

In [34]:
# Scan linearly the collection of document to clustering them using the Jaccard distance = 1 - JaccardSimilarity
# pseudo code:
# Stream_cluster(SortedCollection, Radius)
# C = ∅
# for each d in SortedCollection
# Dist_c = Min (JaccardDistance(c, d), for each medoid c in C)
# if (Dist_c < Radius) then
# add d to cluster c
# else
# make d a new medoid, and add it to C: C = C ∪ d
# return C

In [35]:
from sklearn.metrics import jaccard_score

# def jaccard_distance(x, y):
#     intersection = np.sum(np.minimum(x, y))
#     union = np.sum(np.maximum(x, y))
#     return 1 - intersection / union if union > 0 else 0

# def stream_cluster(SortedCollection, Radius, k):
#     C = []
#     Clusters = {}

#     for d in SortedCollection:
#             if C:
#                 # Calcoli la lista delle distanze una volta sola
#                 distances = [1 - jaccard_score(list(c), list(d), average='binary') for c in C]
                
#                 # Estrai indice e valore minimo senza ricalcolare
#                 cluster = np.argmin(distances)
#                 dist_c = distances[cluster]
#             else:
#                 cluster, dist_c = 1, 0

#     if dist_c < Radius:
#         Clusters[cluster].append(d)
#     else:
#         # make d a new medoid, and add it to C
#         C.append(d)
#     return C

from collections import defaultdict
from sklearn.metrics import jaccard_score
import numpy as np

# def stream_cluster(SortedCollection, Radius, k): 
#     C = []                       # Conterrà le liste di term_id dei medoidi
#     medoid_sets = {}
#     Clusters = defaultdict(list) # Conterrà gli ID (chiavi) dei documenti suddivisi per cluster

#     # Iteriamo usando .items() per avere sia l'ID del documento che la sua lista di termini
#     for doc_id, doc_terms in SortedCollection.items():
        
#         set_d = set(doc_terms)
#         # Se C è vuoto, il primo documento diventa il primo medoide
#         if not C:
#             Clusters[0].append(doc_id) # Salva l'ID nel cluster 0
#             medoid_sets[doc_id] = set_d
#             C.append(doc_id)
#             continue

#         # Calcolo delle distanze usando doc_terms
#         distances = []
#         for c in C:
#             # print(SortedCollection)
#             # centroid_set = set(SortedCollection.get(c))
#             # set_c = set(c) # Recuperiamo i termini del medoide tramite il suo ID 'c'

#             centroid_set = medoid_sets[c]
            
#             # Formula di Jaccard: 1 - (Intersezione / Unione)
#             intersection = len(centroid_set.intersection(set_d))
#             union = len(centroid_set.union(set_d))
            
#             jaccard_dist = 1 - (intersection / union) if union > 0 else 1
#             distances.append(jaccard_dist)
        
#         # print(distances)
#         cluster = np.argmin(distances)
#         dist_c = distances[cluster]

#         if dist_c < Radius or len(C) >= k:
#             Clusters[cluster].append(doc_id)
#         else:
#             C.append(doc_id)
#             medoid_sets[doc_id] = set_d
#             Clusters[len(C)-1].append(doc_id)
            
#     return C, Clusters

In [36]:
from collections import defaultdict
import numpy as np

def stream_cluster(SortedCollection, Radius, k): 
    C = []                       # Lista dei doc_id dei medoidi
    medoid_sets = {}             # medoid_id -> set di termini
    Clusters = defaultdict(list) # medoid_id -> lista di doc_id appartenenti a quel cluster

    for doc_id, doc_terms in SortedCollection.items():
        set_d = set(doc_terms)
        
        # Se non ci sono ancora medoidi, il primo documento lo diventa di diritto
        if not C:
            C.append(doc_id)
            medoid_sets[doc_id] = set_d
            Clusters[doc_id].append(doc_id)
            continue

        # Calcolo delle distanze da tutti i medoidi correnti
        distances = []
        for c in C:
            centroid_set = medoid_sets[c]
            intersection = len(centroid_set.intersection(set_d))
            union = len(centroid_set.union(set_d))
            jaccard_dist = 1 - (intersection / union) if union > 0 else 1
            distances.append(jaccard_dist)
        
        # Trova il medoide più vicino
        min_idx = np.argmin(distances)
        dist_c = distances[min_idx]
        closest_medoid = C[min_idx]

        # Se è abbastanza vicino OPPURE abbiamo raggiunto il numero massimo di cluster k
        if dist_c < Radius or len(C) >= k:
            Clusters[closest_medoid].append(doc_id)
        else:
            # Crea un nuovo cluster con questo documento come medoide
            C.append(doc_id)
            medoid_sets[doc_id] = set_d
            Clusters[doc_id].append(doc_id)
            
    return C, Clusters

In [37]:
# TODO vary for k!! take in consideration the execution time and 
C, Clusters = stream_cluster(doc_id_term_id, 0.7,100)
print(Clusters)

defaultdict(<class 'list'>, {1404: [1404, 1407, 1413, 1412, 1405, 5304, 2109, 3574, 521, 4674, 3451, 8373, 914, 5665, 5904, 6304, 7757, 4155, 4994, 7085, 7843, 4677, 593, 599, 1516, 2509, 2172, 9499, 3578, 8319, 5346, 9492, 3465, 2428, 3211, 4599, 4911, 244, 3473, 422, 7707, 2737, 8177, 7038, 8366, 5107, 9201, 3626, 5526, 6392, 9206, 5023, 7104, 6980, 2429, 5945, 3257, 5944, 3415, 5946, 5948, 3706, 6389], 5539: [5539, 9340, 5531, 5583, 9578, 9776, 6293, 6495], 9783: [9783, 9785, 9789, 9784, 9786, 9790, 9791, 9792, 9787, 3949, 6044, 3952, 3950, 8298, 121, 6175, 4201, 5235, 2615, 3779, 4504, 4875, 7506, 5724, 8979, 4106, 8104, 5259, 607, 1095, 2142, 3771, 1332, 2794, 8164, 2433, 2623, 3951, 3954, 3956, 7505, 8282, 8590, 789, 2260, 4912, 1090, 6124, 8598, 9252, 9532, 27, 6149, 9586, 3058], 4453: [4453, 4456, 4311, 4459, 4077, 1005, 4457, 4461, 4455, 2288, 9279, 4458, 5875, 9284, 1369, 4194, 9324, 4187, 2859, 2377, 2856, 9760, 4831, 2857, 3424, 9391, 3433, 6015, 3242, 6817, 8059, 8943, 213

## Kmeans + TCP

In [38]:
# from sklearn.cluster import KMeans, DBSCAN
# from sklearn.preprocessing import normalize

# def jaccard_distance(x, y):
#     intersection = np.sum(np.minimum(x, y))
#     union = np.sum(np.maximum(x, y))
#     return 1 - intersection / union if union > 0 else 0

# clustering = DBSCAN(metric=jaccard_distance, eps=0.5, min_samples=2)

# # norm_df = normalize(df, norm="l2") #! cosa fa , è fondamentele se non la metti clusteing fa schifo 

# # kmeans = KMeans(n_clusters=K, random_state=0) #! use jaccard distance instead of euclidean

# clusters = clustering.fit(doc_id_term_id.values())
# print("Cluster centers:", clustering.cluster_centers_)

# document_clusters = clustering.labels_
# print("Document clusters:", document_clusters)

## Medoids

In [39]:
# medoids = []

# for i in range(kmeans.n_clusters):
#     cluster_docs = np.flatnonzero(document_clusters == i)
#     cluster_vectors = norm_df[cluster_docs]
#     medoid_idx = np.argmin(np.linalg.norm(cluster_vectors - kmeans.cluster_centers_[i], axis=1))
#     medoids.append(cluster_docs[medoid_idx])



# medoid_vectors = norm_df[medoids]
# nonzero_cols = medoid_vectors.getnnz(axis=0) > 0
# medoid_vectors = medoid_vectors[:, nonzero_cols].toarray()


In [40]:
from sklearn.metrics import pairwise_distances
import numpy as np

# def calculate_distance_matrix(medoid_vectors, method="euclidean"):
    
#     dist = np.zeros((len(medoid_vectors), len(medoid_vectors)))

#     for i, v1 in enumerate(medoid_vectors):
#         set_v1 = set(v1)

#         for j, v2 in enumerate(medoid_vectors):

#             if(i == j):
#                 continue
            
#             set_v2 = set(v2)            
#             # Formula di Jaccard: 1 - (Intersezione / Unione)
#             intersection = len(set_v1.intersection(set_v2))
#             union = len(set_v1.union(set_v2))
            
#             jaccard_dist = 1 - (intersection / union) if union > 0 else 1
#             dist[i][j] = jaccard_dist
        
#     return dist

# calcolo solo la triangolare superiore
def calculate_distance_matrix(medoid_vectors):
    n = len(medoid_vectors)
    dist = np.zeros((n, n))
    
    # Trasformiamo in set una volta sola prima dei cicli
    sets = [set(v) for v in medoid_vectors]

    for i in range(n):
        for j in range(i + 1, n):  # Parte da i+1 per sfruttare la simmetria
            intersection = len(sets[i].intersection(sets[j]))
            union = len(sets[i].union(sets[j]))
            
            jaccard_dist = 1 - (intersection / union) if union > 0 else 1
            
            dist[i][j] = jaccard_dist
            dist[j][i] = jaccard_dist  # Specchia il risultato
            
    return dist

    
    # if method == "euclidean":
    #     return pairwise_distances(medoid_vectors, metric="euclidean")
    # elif method == "cosine":
    #     return pairwise_distances(medoid_vectors, metric="cosine")
    # elif method == "jaccard":
    #     # Jaccard in sklearn works on boolean/binary features
    #     return pairwise_distances(medoid_vectors, metric="jaccard")
    # else:
    #     raise ValueError(f"Unknown method: {method}")

In [41]:
medoid_vectors = [doc_id_term_id.get(c) for c in C]

In [42]:
dist_matrix = calculate_distance_matrix(medoid_vectors)

In [43]:
import networkx as nx
tsp = nx.approximation.traveling_salesman_problem

tsp_graph = nx.from_numpy_array(dist_matrix)
tsp_path = tsp(tsp_graph, cycle=False)
print("TSP path (medoid indices):", tsp_path)

TSP path (medoid indices): [65, 7, 90, 81, 92, 9, 18, 52, 29, 64, 91, 10, 50, 25, 83, 67, 60, 51, 2, 98, 41, 44, 16, 57, 5, 85, 4, 3, 94, 61, 11, 56, 31, 70, 6, 97, 1, 95, 88, 21, 77, 86, 30, 0, 55, 89, 40, 39, 80, 75, 74, 19, 13, 38, 93, 26, 42, 99, 63, 48, 20, 82, 23, 68, 62, 17, 33, 35, 14, 54, 78, 69, 22, 59, 84, 32, 58, 12, 8, 15, 76, 34, 45, 96, 49, 72, 28, 73, 66, 46, 79, 27, 47, 43, 24, 37, 87, 36, 71, 53]


In [44]:
# def reorder_documents_by_tsp(document_clusters, medoids, tsp_path):
#     """
#     Reorder documents by cluster order induced by TSP over medoids.
#     Inside each cluster, keep documents in their original order (row order).

#     Args:
#         document_clusters: array-like of shape (n_docs,), cluster label per doc.
#         medoids: list of medoid doc indices (not required for ordering here, kept for API compatibility).
#         tsp_path: list of cluster ids in the order returned by TSP.

#     Returns:
#         ordered_doc_ids: list of document row indices in final order.
#         cluster_to_docs: dict {cluster_id: [doc_ids in original order]}.
#     """
#     document_clusters = np.asarray(document_clusters)

#     # Build cluster -> docs preserving natural/original order
#     cluster_to_docs = {
#         c: np.flatnonzero(document_clusters == c).tolist()
#         for c in range(len(medoids))
#     }

#     # Concatenate clusters following TSP order
#     ordered_doc_ids = []
#     for c in tsp_path:
#         ordered_doc_ids.extend(cluster_to_docs.get(c, []))

#     return ordered_doc_ids, cluster_to_docs

In [45]:
# ordered_doc_ids, cluster_docs = reorder_documents_by_tsp(Clusters, C, tsp_path)

In [46]:
# def reorder_documents_by_tsp_new(tsp_path, cluster, C):

#     tot = []
#     for i in tsp_path:
#         docs = cluster.get(i)
#         for d in docs:  
#             tot.append(d)
#     return tot

def reorder_documents_by_tsp(tsp_path, cluster, C):
    tot = []
    for i in tsp_path:
        # 1. Recuperiamo il vero DocID del medoide usando l'indice del TSP
        medoid_id = C[i] 
        
        # 2. Accediamo al dizionario dei cluster usando il DocID del medoide
        docs = cluster.get(medoid_id, [])
        
        # 3. Aggiungiamo tutti i documenti di questo cluster alla lista finale
        for d in docs:  
            tot.append(d)
            
    return tot

reord_docids = reorder_documents_by_tsp(tsp_path,Clusters,C)
print(min(reord_docids))

print("Numero totale di documenti riordinati:", len(reord_docids))
print("Minimo DocID trovato (dovrebbe essere coerente con la collezione):", min(reord_docids))

1
Numero totale di documenti riordinati: 10000
Minimo DocID trovato (dovrebbe essere coerente con la collezione): 1


In [47]:
from itertools import islice

docs = list(islice(msmarco.docs_iter(), NUM_DOCS))

class OrderedDataset:
    def __init__(self, docs, order):
        self.docs = docs
        self.order = order
    def docs_iter(self):
        for i in self.order:
            yield self.docs[i-1]

ordered_dataset = OrderedDataset(docs, reord_docids)
reordered_inverted_index = Index()
reordered_inverted_index.build_index(
    ordered_dataset,
    tokenizer,
    max_num_docs=-1, #! -1 since reordered docs are already limited to 10001 in the original index
    d_gap=True
)

stat_r = reordered_inverted_index.get_approx_size()

unordered_index = Index()
unordered_index.build_index(random_dataset, tokenizer, max_num_docs=NUM_DOCS, d_gap=True)

stat_m = unordered_index.get_approx_size()
print(f"Original index size: {stat_m} bytes")
print(f"Reordered index size: {stat_r} bytes")

ind_1 = reordered_inverted_index.index
ind_2 = unordered_index.index

print("Msmarco index size:", len(unordered_index.index))


=== Index Size Breakdown ===
Doc ID bits:  12,172,320
TF bits:      12,172,320
Total bits:   24,344,640
Total bytes:  3,043,080.00

=== Index Size Breakdown ===
Doc ID bits:  12,172,320
TF bits:      12,172,320
Total bits:   24,344,640
Total bytes:  3,043,080.00
Original index size: {'term_bytes': 0, 'doc_id_bits': 12172320, 'tf_bits': 12172320, 'total_bits': 24344640, 'total_bytes': 3043080.0} bytes
Reordered index size: {'term_bytes': 0, 'doc_id_bits': 12172320, 'tf_bits': 12172320, 'total_bits': 24344640, 'total_bytes': 3043080.0} bytes
Msmarco index size: 37675


In [48]:
def all_gaps(index):
    return [
        gap
        for posting_list in index.index.values()
        for gap in posting_list
    ]

gaps_order = all_gaps(reordered_inverted_index)

gaps_unorderd = all_gaps(unordered_index)


print("AVG ordered:", np.mean(gaps_order))
print("AVG unordered:", np.mean(gaps_unorderd))

print("MEDIAN ordered:", np.median(gaps_order))
print("MEDIAN unordered:", np.median(gaps_unorderd))

AVG ordered: 319.36239599353286
AVG unordered: 322.0252441605216
MEDIAN ordered: 2.0
MEDIAN unordered: 2.0


In [49]:
# Test compressions
import copy

def prepare_index(src_index, method):
    idx = copy.deepcopy(src_index)
    idx.remove_tf()
    idx.compress_index(method)
    return idx

# Original index variants
gamma_index = prepare_index(unordered_index, "gamma")
delta_index = prepare_index(unordered_index, "delta")
vb_index = prepare_index(unordered_index, "varbyte")

# Reordered index variants
gamma_index_r = prepare_index(reordered_inverted_index, "gamma")
delta_index_r = prepare_index(reordered_inverted_index, "delta")
vb_index_r = prepare_index(reordered_inverted_index, "varbyte")

print("\n" + "=" * 70)
print(f"{'ORIGINAL (NOT REORDERED) INDEX':^70}")
print("=" * 70)
print("Uncompressed:")
unordered_index.remove_tf()  # Ensure TF is removed for size calculation
unordered_index.get_approx_size()

print("\nGamma Compressed:")
gamma_index.get_approx_size()

print("\nDelta Compressed:")
delta_index.get_approx_size()

print("\nVarbyte Compressed:")
vb_index.get_approx_size()

print("\n" + "=" * 70)
print(f"{'REORDERED INDEX (TSP)':^70}")
print("=" * 70)
print("Uncompressed:")
reordered_inverted_index.remove_tf()  # Ensure TF is removed for size calculation
reordered_inverted_index.get_approx_size()

print("\nGamma Compressed:")
gamma_index_r.get_approx_size()

print("\nDelta Compressed:")
delta_index_r.get_approx_size()

print("\nVarbyte Compressed:")
vb_index_r.get_approx_size()


                    ORIGINAL (NOT REORDERED) INDEX                    
Uncompressed:

=== Index Size Breakdown ===
Doc ID bits:  12,172,320
TF bits:      0
Total bits:   12,172,320
Total bytes:  1,521,540.00

Gamma Compressed:

=== Index Size Breakdown ===
Doc ID bits:  4,398,043
TF bits:      0
Total bits:   4,398,043
Total bytes:  549,755.38

Delta Compressed:

=== Index Size Breakdown ===
Doc ID bits:  3,898,538
TF bits:      0
Total bits:   3,898,538
Total bytes:  487,317.25

Varbyte Compressed:

=== Index Size Breakdown ===
Doc ID bits:  4,228,216
TF bits:      0
Total bits:   4,228,216
Total bytes:  528,527.00

                        REORDERED INDEX (TSP)                         
Uncompressed:

=== Index Size Breakdown ===
Doc ID bits:  12,172,320
TF bits:      0
Total bits:   12,172,320
Total bytes:  1,521,540.00

Gamma Compressed:

=== Index Size Breakdown ===
Doc ID bits:  4,111,661
TF bits:      0
Total bits:   4,111,661
Total bytes:  513,957.62

Delta Compressed:

=== Inde

{'term_bytes': 0,
 'doc_id_bits': 4128904,
 'tf_bits': 0,
 'total_bits': 4128904,
 'total_bytes': 516113.0}

In [ ]:
# divider = "#" * 100

# print(divider)
# print("reordered")
# reordered_inverted_index.compress_index("gamma")
# print(reordered_inverted_index.get_approx_size())
# reordered_inverted_index.save_index(file_path="reord.bin")

# print(divider)
# print("not reordered")
# unordered_index.compress_index("gamma")
# print(unordered_index.get_approx_size())
# unordered_index.save_index(file_path="not_reord.bin")

####################################################################################################
reordered

=== Index Size Breakdown ===
Doc ID bits:  4,111,661
TF bits:      583,087
Total bits:   4,694,748
Total bytes:  586,843.50
{'term_bytes': 0, 'doc_id_bits': 4111661, 'tf_bits': 583087, 'total_bits': 4694748, 'total_bytes': 586843.5}
####################################################################################################
not reordered

=== Index Size Breakdown ===
Doc ID bits:  9,022,855
TF bits:      583,087
Total bits:   9,605,942
Total bytes:  1,200,742.75
{'term_bytes': 0, 'doc_id_bits': 9022855, 'tf_bits': 583087, 'total_bits': 9605942, 'total_bytes': 1200742.75}


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (380385, 2) + inhomogeneous part.